# Nifty 500 Strategy Scanner

Runs the 10-step trend-breakout-retest pipeline across the Nifty 500 universe using free Yahoo Finance data (via `yfinance`).

**Before running:** make sure the following files are in the SAME FOLDER as this notebook:
`pipeline.py`, `indicators.py`, `trend_filter.py`, `consolidation.py`, `breakout.py`, `momentum.py`, `volume_confirm.py`, `entry_retest.py`, `risk_management.py`

Run the cells in order, top to bottom.

## Step 0 — Point Python to your "Market Data" folder

This makes sure Python can find `pipeline.py` and the other strategy files, regardless of where this notebook itself is saved. Run this FIRST, before anything else.

In [ ]:
import os
import sys
import glob

# --- Robust auto-locate: finds pipeline.py even if Desktop is redirected
# (common on Windows with OneDrive) or the folder name differs slightly ---

home = os.path.expanduser("~")

candidate_paths = [
    os.path.join(home, "Desktop", "Market Data"),
    os.path.join(home, "OneDrive", "Desktop", "Market Data"),
]
# Also catch OneDrive folders named like "OneDrive - CompanyName"
candidate_paths += glob.glob(os.path.join(home, "OneDrive*", "Desktop", "Market Data"))

found_folder = None
for path in candidate_paths:
    if os.path.isfile(os.path.join(path, "pipeline.py")):
        found_folder = path
        break

# If none of the obvious guesses worked, search the whole home folder
# (this can take a few seconds but is thorough)
if not found_folder:
    print("Not found in common locations - searching your whole user folder for pipeline.py...")
    for root, dirs, files in os.walk(home):
        # skip huge/irrelevant system folders to keep this fast
        dirs[:] = [d for d in dirs if d not in (
            "Library", "AppData", "node_modules", ".git", "anaconda3", "miniconda3"
        )]
        if "pipeline.py" in files:
            found_folder = root
            break

if found_folder:
    os.chdir(found_folder)
    sys.path.insert(0, found_folder)
    print(f"FOUND pipeline.py in: {found_folder}")
    print(f"Working directory set to: {os.getcwd()}")
    print("\nFiles in this folder:")
    for f in sorted(os.listdir(found_folder)):
        print(f"  - {f}")
else:
    print("COULD NOT FIND pipeline.py anywhere in your user folder.")
    print("This means the file may not have actually been saved/downloaded yet,")
    print("or it's in a location this search didn't cover (e.g. an external drive).")
    print("\nRun this in a new cell to search manually:")
    print('  for root, dirs, files in os.walk("/"):')
    print('      if "pipeline.py" in files: print(root)')
    print("\n(Note: searching from '/' can be slow - Ctrl+C / Interrupt Kernel to stop it.)")

FOUND pipeline.py in: /Users/sairitdey/Desktop/Market Data
Working directory set to: /Users/sairitdey/Desktop/Market data

Files in this folder:
  - .DS_Store
  - .ipynb_checkpoints
  - Data colllection
  - __pycache__
  - breakout.py
  - consolidation.py
  - entry_retest.py
  - files.zip
  - indicators.py
  - momentum.py
  - mtf_swing_strategy_v2.ipynb
  - mtf_swing_strategy_v2.py
  - nifty500_scan_results.csv
  - nifty500_scanner.ipynb
  - pipeline.py
  - risk_management.py
  - scan_nifty500.py
  - trading_screener.py
  - trend_filter.py
  - universe_cache
  - volume_confirm.py


## Step 1 — Install required packages
Run this once. If already installed, it's safe to skip or re-run.

In [ ]:
!pip install yfinance pandas requests --quiet

## Step 2 — Check that the strategy modules are found
This confirms `pipeline.py` and its dependencies are in the same folder as this notebook.

In [ ]:
import os

required_files = [
    "pipeline.py", "indicators.py", "trend_filter.py", "consolidation.py",
    "breakout.py", "momentum.py", "volume_confirm.py", "entry_retest.py",
    "risk_management.py",
]

missing = [f for f in required_files if not os.path.exists(f)]

if missing:
    print("MISSING FILES - copy these into this notebook's folder before continuing:")
    for f in missing:
        print(f"  - {f}")
else:
    print("All strategy modules found. You're good to continue.")

All strategy modules found. You're good to continue.


## Step 3 — Imports

In [ ]:
import time
import io
import requests
import pandas as pd
import yfinance as yf

from pipeline import run_pipeline

pd.set_option('display.max_rows', 100)
pd.set_option('display.width', 150)

## Step 4 — Get the Nifty 500 symbol list
Tries to fetch the live list from NSE. NSE often blocks non-browser requests, so a fallback list is used if that fails - edit `FALLBACK_SYMBOLS` below with the full list (downloaded manually from NSE) for a complete scan.

In [ ]:
NIFTY500_LIST_URL = "https://archives.nseindia.com/content/indices/ind_nifty500list.csv"

FALLBACK_SYMBOLS = [
    "RELIANCE", "TCS", "HDFCBANK", "INFY", "ICICIBANK", "HINDUNILVR",
    "ITC", "SBIN", "BHARTIARTL", "BAJFINANCE", "HINDZINC", "TATASTEEL",
    # Extend this list with more Nifty 500 symbols as needed
]

def get_nifty500_symbols():
    try:
        headers = {"User-Agent": "Mozilla/5.0"}
        resp = requests.get(NIFTY500_LIST_URL, headers=headers, timeout=10)
        resp.raise_for_status()
        df = pd.read_csv(io.StringIO(resp.text))
        symbols = df["Symbol"].tolist()
        print(f"Fetched {len(symbols)} symbols from NSE.")
        return symbols
    except Exception as e:
        print(f"Could not fetch NSE list automatically ({e}).")
        print(f"Using fallback list of {len(FALLBACK_SYMBOLS)} symbols instead.")
        return FALLBACK_SYMBOLS

symbols = get_nifty500_symbols()
print(f"\nTotal symbols to scan: {len(symbols)}")

Fetched 500 symbols from NSE.

Total symbols to scan: 500


## Step 5 — OHLCV fetch function (Yahoo Finance)

In [ ]:
def fetch_ohlcv_yfinance(symbol, period="90d", interval="60m"):
    ticker = f"{symbol}.NS"
    data = yf.download(ticker, period=period, interval=interval, progress=False)

    if data.empty:
        raise ValueError(f"No data returned for {ticker}")

    # yfinance returns MultiIndex columns like ('Close', 'RELIANCE.NS') -
    # flatten to just the field name before lowercasing.
    if isinstance(data.columns, pd.MultiIndex):
        data.columns = data.columns.get_level_values(0)

    data = data.rename(columns=str.lower)
    data = data[["open", "high", "low", "close", "volume"]]
    return data

## Step 6 — Quick test on a single stock
Run this first to sanity-check everything works before scanning all 500.

In [ ]:
test_df = fetch_ohlcv_yfinance("RELIANCE")
print(test_df.shape)
test_df.tail()

(610, 5)


Price,open,high,low,close,volume
Datetime,,,,,
2026-09-03 15:15:00+05:30,1309.099976,1309.099976,1302.500000,1302.500000,690251
2026-09-04 09:15:00+05:30,1304.099976,1332.199951,1304.099976,1327.000000,3550780
2026-09-04 10:15:00+05:30,1326.800049,1328.900024,1325.099976,1325.500000,1233518
2026-09-04 11:15:00+05:30,1325.500000,1333.000000,1325.000000,1330.099976,2137828
2026-09-04 12:15:00+05:30,1330.199951,1332.400024,1329.099976,1329.099976,831414


In [ ]:
test_signals = run_pipeline(test_df, symbol="RELIANCE")
print(f"Signals found: {len(test_signals)}")
test_signals

Signals found: 0


[]

## Step 7 — Full scan function

In [ ]:
def scan(symbols, cutoff_date=None, freshness_days=3, drop_incomplete_candle=True):
    """
    cutoff_date=None -> automatically uses TODAY'S date (IST), every run.
    Pass an explicit "YYYY-MM-DD" string to check a specific past date instead.
    drop_incomplete_candle=True -> removes the most recent candle if it's
    still forming (hasn't hit the full hour yet), so you don't act on a
    signal that could still change shape before the candle closes.
    """
    if cutoff_date is None:
        cutoff = pd.Timestamp.now(tz="Asia/Kolkata").normalize()
    else:
        cutoff = pd.Timestamp(cutoff_date)

    fresh_cutoff = cutoff - pd.Timedelta(days=freshness_days)

    all_signals = []

    for i, symbol in enumerate(symbols):
        try:
            df = fetch_ohlcv_yfinance(symbol)

            # Yahoo returns timezone-aware timestamps (Asia/Kolkata).
            if df.index.tz is not None:
                cutoff_tz = cutoff.tz_localize(df.index.tz) if cutoff.tz is None else cutoff.tz_convert(df.index.tz)
                fresh_cutoff_tz = fresh_cutoff.tz_localize(df.index.tz) if fresh_cutoff.tz is None else fresh_cutoff.tz_convert(df.index.tz)
            else:
                cutoff_tz = cutoff
                fresh_cutoff_tz = fresh_cutoff

            # Drop the current in-progress hourly candle if it hasn't closed yet.
            if drop_incomplete_candle and len(df) > 0:
                now_tz = pd.Timestamp.now(tz="Asia/Kolkata")
                now_compare = now_tz.tz_convert(df.index.tz) if df.index.tz is not None else now_tz.tz_localize(None)
                last_candle_start = df.index[-1]
                candle_end = last_candle_start + pd.Timedelta(hours=1)
                if candle_end > now_compare:
                    df = df.iloc[:-1]

            df = df[df.index <= cutoff_tz]  # avoid look-ahead past cutoff

            signals = run_pipeline(df, symbol=symbol)
            fresh = [s for s in signals if fresh_cutoff_tz <= s["date"] <= cutoff_tz]
            all_signals.extend(fresh)

            status = f"{len(fresh)} fresh signal(s)" if fresh else "no signal"
            print(f"[{i+1}/{len(symbols)}] {symbol}: {status}")

        except Exception as e:
            print(f"[{i+1}/{len(symbols)}] {symbol}: ERROR - {e}")

        time.sleep(0.5)  # avoid Yahoo rate limiting

    if not all_signals:
        print("\nNo stocks currently pass the full 10-step criteria.")
        return pd.DataFrame()

    result_df = pd.DataFrame(all_signals)
    result_df.sort_values("date", ascending=False, inplace=True)
    return result_df.reset_index(drop=True)

## Step 8 — Run the full scan
This will take a few minutes depending on how many symbols are in your list (rate-limited to be polite to Yahoo's servers).

In [ ]:
# cutoff_date=None -> uses today's date automatically, every run.
# Pass cutoff_date="2026-08-28" (or any date) to check a specific past day instead.
results = scan(symbols, cutoff_date=None, freshness_days=3)

[1/500] 360ONE: no signal
[2/500] 3MINDIA: no signal
[3/500] ABB: no signal
[4/500] ACC: no signal
[5/500] ACMESOLAR: no signal
[6/500] AIAENG: no signal
[7/500] APLAPOLLO: no signal
[8/500] AUBANK: no signal
[9/500] AWL: no signal
[10/500] AADHARHFC: no signal
[11/500] AARTIIND: no signal
[12/500] AAVAS: no signal
[13/500] ABBOTINDIA: no signal
[14/500] ACE: no signal
[15/500] ACUTAAS: no signal
[16/500] ADANIENSOL: no signal
[17/500] ADANIENT: no signal
[18/500] ADANIGREEN: no signal
[19/500] ADANIPORTS: no signal
[20/500] ADANIPOWER: no signal
[21/500] ATGL: no signal
[22/500] ABCAPITAL: no signal
[23/500] ABFRL: no signal
[24/500] ABLBL: no signal
[25/500] ABREL: no signal
[26/500] ABSLAMC: no signal
[27/500] CPPLUS: no signal
[28/500] AEGISLOG: no signal
[29/500] AEGISVOPAK: no signal
[30/500] AFCONS: 1 fresh signal(s)
[31/500] AFFLE: no signal
[32/500] AJANTPHARM: no signal
[33/500] ALKEM: no signal
[34/500] ABDL: no signal
[35/500] ARE&M: no signal
[36/500] AMBER: no signal
[37/

$JWL.NS: possibly delisted; no price data found  (period=90d)

1 Failed download:
['JWL.NS']: possibly delisted; no price data found  (period=90d)


[275/500] JWL: ERROR - No data returned for JWL.NS
[276/500] JYOTICNC: no signal
[277/500] KPRMILL: no signal
[278/500] KEI: no signal
[279/500] KPITTECH: no signal
[280/500] KAJARIACER: no signal
[281/500] KPIL: no signal
[282/500] KALYANKJIL: no signal
[283/500] KARURVYSYA: no signal
[284/500] KAYNES: no signal
[285/500] KEC: no signal
[286/500] KFINTECH: no signal
[287/500] KIRLOSENG: no signal
[288/500] KOTAKBANK: no signal
[289/500] KIMS: no signal
[290/500] LTF: no signal
[291/500] LTTS: no signal
[292/500] LGEINDIA: no signal
[293/500] LICHSGFIN: no signal
[294/500] LTFOODS: no signal
[295/500] LTM: no signal
[296/500] LT: no signal
[297/500] LATENTVIEW: no signal
[298/500] LAURUSLABS: no signal
[299/500] THELEELA: no signal
[300/500] LEMONTREE: no signal
[301/500] LENSKART: no signal
[302/500] LICI: no signal
[303/500] LINDEINDIA: no signal
[304/500] LLOYDSME: no signal
[305/500] LODHA: no signal
[306/500] LUPIN: no signal
[307/500] MMTC: no signal
[308/500] MRF: no signal
[309

## Diagnostic — confirm what date this actually ran against
Run this right after Step 8 to see exactly what 'now' and 'cutoff' the scan used.

In [ ]:
import pandas as pd
print('Current real time (IST):', pd.Timestamp.now(tz="Asia/Kolkata"))
print('Most recent date in your results:', results['date'].max() if not results.empty else 'no results')
print('\nIf the current time above is TODAY but results only show old dates,')
print('it means Step 8 was run earlier and not re-executed since - re-run Step 8.')

Current real time (IST): 2026-09-04 13:07:26.137837+05:30
Most recent date in your results: 2026-09-03 14:15:00+05:30

If the current time above is TODAY but results only show old dates,
it means Step 8 was run earlier and not re-executed since - re-run Step 8.


## Step 9 — View results

In [ ]:
if not results.empty:
    display(results)
else:
    print("No signals to display.")

,symbol,date,entry_mode,entry,stop_loss,target_1,target_2,risk_pct,atr,rvol,volume_grade,breakout_atr
0,TATACAP,2026-09-03 14:15:00+05:30,IMMEDIATE,372.55,368.82,377.13,380.57,1.0,2.29,2.14,STRONG (>=2x avg),0.70
1,AFCONS,2026-09-03 11:15:00+05:30,IMMEDIATE,295.25,292.30,300.53,304.49,1.0,2.64,6.19,STRONG (>=2x avg),2.88


## Step 10 — Save results to CSV (optional)

In [ ]:
if not results.empty:
    results.to_csv("nifty500_scan_results.csv", index=False)
    print("Saved to nifty500_scan_results.csv")

Saved to nifty500_scan_results.csv
